# Disjoint trial splitting — the electrode-definition ↔ decoding circularity control

**The trap.** When decoding is restricted to a *selected* electrode set, and that
selection is computed on the **same trials** the decoder then scores, the
selection biases the decoding accuracy upward. This is **double-dipping**
(circular analysis): the "test" set already leaked into the "selection" step.
See the analysis plan §0.1/§0.2 and
`docs/decoding_and_electrode_definition_notes.md` §C.

**The fix.** A disjoint trial partition:

1. split each subject's trials into a **definition** set (`P_def`) and a
   **decode** set (`P_dec`), stratified so both stay balanced,
2. define / select electrodes on `P_def` **only**,
3. decode on `P_dec` **only**, restricted to those electrodes.

Because the selection never sees the trials the accuracy is computed on, it
cannot inflate it. This notebook walks the pure primitives in
`src/analysis/decoding/trial_splitting.py`
(`stratified_trial_split`, `strata_key_from_metadata`,
`select_responsive_channels`), *demonstrates the double-dipping bias and how the
split removes it*, and then shows the pipeline orchestration helper
`apply_electrode_definition_split`.

Everything except the final (MNE) cell runs on synthetic data with no cluster
access. The primitives are unit-tested in
`tests/analysis/decoding/test_trial_splitting.py` (16 tests).

In [ ]:
import os, sys
# make the project importable no matter where the notebook is launched from
here = os.getcwd()
root = here
for _ in range(6):
    if os.path.exists(os.path.join(root, 'src', 'analysis')):
        break
    root = os.path.dirname(root)
if root not in sys.path:
    sys.path.insert(0, root)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis.decoding import trial_splitting as ts
print("loaded trial_splitting from", ts.__file__)

## 1. `stratified_trial_split` — disjoint, balanced halves

Split trial indices into two disjoint sets, holding a fixed fraction of **every
stratum** on the definition side. Stratifying on the condition/block keeps both
halves balanced and stops either from being temporally lopsided.

In [ ]:
# 90 trials: a 2 (congruency) x 2 (block) design, unevenly sized cells.
rng = np.random.RandomState(0)
strata = np.array(
    ["c|mostly_con"] * 30 + ["i|mostly_con"] * 15
    + ["c|mostly_inc"] * 15 + ["i|mostly_inc"] * 30
)
idx_def, idx_dec = ts.stratified_trial_split(strata, frac_def=0.5, seed=0)

# (a) the two sets are disjoint and together cover every trial exactly once
assert set(idx_def).isdisjoint(set(idx_dec))
assert sorted(np.concatenate([idx_def, idx_dec])) == list(range(len(strata)))
print(f"def: {len(idx_def)} trials | dec: {len(idx_dec)} trials | disjoint & complete ✓")

# (b) each stratum is split ~in half on BOTH sides -> balanced
bal = pd.DataFrame({
    "stratum": pd.unique(strata),
}).set_index("stratum")
bal["n_total"] = [np.sum(strata == s) for s in bal.index]
bal["n_def"] = [np.sum(strata[idx_def] == s) for s in bal.index]
bal["n_dec"] = [np.sum(strata[idx_dec] == s) for s in bal.index]
bal

The split is **deterministic** under `seed` (so a run reproduces), and a
**singleton** stratum is never silently dropped — it goes to the definition
side.

In [ ]:
# deterministic: same seed -> identical partition
a = ts.stratified_trial_split(strata, frac_def=0.5, seed=7)
b = ts.stratified_trial_split(strata, frac_def=0.5, seed=7)
assert np.array_equal(a[0], b[0]) and np.array_equal(a[1], b[1])
print("same seed -> identical split ✓")

# a singleton stratum lands in the definition set (never dropped)
singleton = np.array([0, 1, 1, 1, 1])   # stratum 0 has one trial
d, _ = ts.stratified_trial_split(singleton, frac_def=0.5, seed=0)
assert 0 in set(d)
print("singleton stratum -> definition set ✓")

# frac_def controls the sizes
d80, r80 = ts.stratified_trial_split(np.zeros(100, dtype=int), frac_def=0.8, seed=0)
print(f"frac_def=0.8 -> {len(d80)} def / {len(r80)} dec")

## 2. `strata_key_from_metadata` — build strata from trial metadata

In the pipeline the stratum key comes from the epochs' `metadata` DataFrame. This
helper joins the requested columns into one key per trial, and **skips missing
columns with a warning** (so a long job degrades to a partial stratification
rather than crashing).

In [ ]:
md_df = pd.DataFrame({
    "congruency": ["c", "i", "c", "i"],
    "switchType": ["s", "s", "r", "r"],
    "blockType":  ["mostly_con", "mostly_con", "mostly_inc", "mostly_inc"],
})
key = ts.strata_key_from_metadata(md_df, ["congruency", "switchType", "blockType"])
print("stratum keys:", list(key))

# a missing column is skipped with a warning; the rest still form the key
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    key2 = ts.strata_key_from_metadata(md_df, ["congruency", "not_a_column"])
    print("warned about missing column:", any(issubclass(x.category, UserWarning) for x in w))
print("keys with the present column only:", list(key2))

# None / all-missing -> a constant key -> an unstratified random split
const = ts.strata_key_from_metadata(md_df, None)
print("None -> constant key (unstratified):", len(set(const)) == 1)

## 3. `select_responsive_channels` — the held-out FDR selector

This is the selector you run on the **definition** partition. It flags channels
whose analysis-window activity differs from baseline (or from 0, for
baseline-rescaled high-gamma), with **Benjamini–Hochberg FDR across channels**.
Dead / zero-variance channels are never selected.

It deliberately uses a *task-responsiveness* contrast (window vs. baseline),
which is approximately orthogonal to any single decode contrast — the disjoint
trials then make the selection strictly independent of the decode accuracy.

In [ ]:
rng = np.random.RandomState(0)
n_trials = 200
# ch0: strong positive window activity (responsive); ch1: centered at 0 (not)
window_means = np.column_stack([
    rng.normal(1.0, 1.0, n_trials),   # responsive
    rng.normal(0.0, 1.0, n_trials),   # not responsive
])
mask = ts.select_responsive_channels(window_means, baseline_means=None, alpha=0.05)
print("one-sample vs 0   -> selected:", mask, "(expect [True, False])")

# paired window-vs-baseline: ch0 rises above its OWN baseline
baseline = rng.normal(0.0, 1.0, (n_trials, 2))
window = baseline.copy()
window[:, 0] += 0.8
mask_paired = ts.select_responsive_channels(window, baseline_means=baseline, alpha=0.05)
print("paired vs baseline -> selected:", mask_paired, "(expect [True, False])")

# a zero-variance (dead/flat) channel is never selected
flat = np.column_stack([np.ones(50) * 5.0, rng.normal(0, 1, 50)])
print("zero-variance ch0 selected?:", bool(ts.select_responsive_channels(flat, alpha=0.05)[0]),
      "(expect False)")

## 4. ⭐ Why disjoint trials matter — the double-dipping demo (the whole point)

Here is the bias the split exists to remove. Take **pure-noise** data — there is
*no* real signal separating the two classes — and many candidate channels. If we
**select** the channels that best separate the classes and then **score** the
classifier on the *same* trials, we get accuracy well **above chance** purely
from fitting noise. If instead we select on `P_def` and score on the disjoint
`P_dec`, accuracy collapses back to chance, where it belongs.

(We use a tiny nearest-centroid classifier so the notebook needs no sklearn.)

In [ ]:
def nearest_centroid_accuracy(X_tr, y_tr, X_te, y_te):
    """Minimal nearest-centroid classifier accuracy (no sklearn needed)."""
    c0 = X_tr[y_tr == 0].mean(axis=0)
    c1 = X_tr[y_tr == 1].mean(axis=0)
    d0 = np.linalg.norm(X_te - c0, axis=1)
    d1 = np.linalg.norm(X_te - c1, axis=1)
    pred = (d1 < d0).astype(int)
    return np.mean(pred == y_te)

def one_sim(seed, n_trials=60, n_channels=200, k=10):
    rng = np.random.RandomState(seed)
    X = rng.normal(0, 1, (n_trials, n_channels))    # PURE NOISE — no real class signal
    y = rng.randint(0, 2, n_trials)

    # --- NAIVE (double-dipping): select channels AND score on the same trials ---
    sep = np.abs(X[y == 1].mean(0) - X[y == 0].mean(0))
    top = np.argsort(sep)[-k:]
    acc_naive = nearest_centroid_accuracy(X[:, top], y, X[:, top], y)

    # --- DISJOINT: select on P_def, score on P_dec (stratified by label) ---
    idx_def, idx_dec = ts.stratified_trial_split(y, frac_def=0.5, seed=seed)
    Xd, yd = X[idx_def], y[idx_def]
    Xt, yt = X[idx_dec], y[idx_dec]
    sep_def = np.abs(Xd[yd == 1].mean(0) - Xd[yd == 0].mean(0))
    top_def = np.argsort(sep_def)[-k:]
    acc_split = nearest_centroid_accuracy(Xd[:, top_def], yd, Xt[:, top_def], yt)
    return acc_naive, acc_split

res = np.array([one_sim(s) for s in range(300)])
naive, split = res[:, 0], res[:, 1]
print(f"NAIVE  (select + score on same trials): mean acc = {naive.mean():.3f}  <-- inflated above 0.5")
print(f"SPLIT  (select on def, score on dec):    mean acc = {split.mean():.3f}  <-- back at chance")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0.2, 1.0, 33)
ax.hist(naive, bins=bins, alpha=0.7, color="#d95f0e", label="naive (double-dipping)")
ax.hist(split, bins=bins, alpha=0.7, color="#2c7fb8", label="disjoint split")
ax.axvline(0.5, color="k", ls="--", lw=1, label="chance")
ax.set(xlabel="decoding accuracy (pure-noise data)", ylabel="# simulations",
       title="Selecting on the trials you score inflates accuracy; a disjoint split fixes it")
ax.legend(); fig.tight_layout(); plt.show()

The orange distribution sits well above chance **even though there is no real
signal** — that is the double-dipping bias, manufactured entirely by selecting on
the same trials that are scored. The blue distribution (select on `P_def`, score
on the disjoint `P_dec`) is centered on chance. On real data the honest,
split accuracy will typically be **lower than the naive one**; that gap is the
bias the control removes.

## 5. `apply_electrode_definition_split` — wiring the split onto the pipeline

The orchestration helper applies all of the above to the decoding pipeline's data
structures:

- `subjects_mne_objects[sub][condition][epochs_key]` → an MNE `Epochs` object,
- `electrodes[roi][sub]` → a list of candidate channel names.

For every `(subject, condition)` it splits the epochs (stratified within the
object by `strata_cols`), selects responsive channels on the pooled **definition**
trials per subject, restricts `electrodes` to those channels, and returns the
**decode** partition so the downstream decoder never sees the definition trials.

This cell needs MNE (it builds a tiny synthetic `EpochsArray`); it is guarded so
the notebook still runs end-to-end without it.

In [ ]:
try:
    import mne
    mne.set_log_level("ERROR")

    def make_epochs(n_trials, ch_names, sfreq=100, n_times=100, responsive_ch=None, seed=0):
        rng = np.random.RandomState(seed)
        info = mne.create_info(ch_names=list(ch_names), sfreq=sfreq, ch_types="eeg")
        tmin = -0.5
        data = rng.normal(0, 1, (n_trials, len(ch_names), n_times))
        times = np.arange(n_times) / sfreq + tmin
        if responsive_ch is not None:
            j = list(ch_names).index(responsive_ch)
            data[:, j, times >= 0.0] += 1.5      # a real post-stimulus response
        metadata = pd.DataFrame({
            "congruency": rng.choice(["c", "i"], n_trials),
            "switchType": rng.choice(["s", "r"], n_trials),
            "blockType":  rng.choice(["mostly_con", "mostly_inc"], n_trials),
        })
        return mne.EpochsArray(data, info, tmin=tmin, metadata=metadata)

    ch_names = ["A1", "A2", "A3", "A4"]
    subjects_mne_objects = {
        "sub1": {
            "cond1": {"HG_ev1_power_rescaled": make_epochs(40, ch_names, responsive_ch="A1", seed=1)},
            "cond2": {"HG_ev1_power_rescaled": make_epochs(36, ch_names, responsive_ch="A1", seed=2)},
        }
    }
    electrodes = {"roiA": {"sub1": ch_names}}     # all four are candidates
    rois = ["roiA"]

    n_before = sum(len(o["HG_ev1_power_rescaled"])
                   for o in subjects_mne_objects["sub1"].values())

    decode_objs, elecs_restricted = ts.apply_electrode_definition_split(
        subjects_mne_objects, electrodes, rois,
        frac_def=0.5, strata_cols=("congruency", "switchType", "blockType"),
        seed=0, alpha=0.05, window_tmin=0.0,
    )

    n_after = sum(len(o["HG_ev1_power_rescaled"])
                  for o in decode_objs["sub1"].values())
    print(f"trials before (def+dec): {n_before}  ->  decode-only trials: {n_after}")
    print("electrodes selected on the DEFINITION partition:", elecs_restricted["roiA"]["sub1"])
    print("(A1 is the planted responsive channel; the noise channels drop out)")
except ImportError:
    print("MNE not installed — skipping the pipeline-orchestration demo.")
    print("On the cluster (conda env 'ieeg') this returns the decode partition")
    print("plus the electrodes reselected on the held-out definition trials.")

## Running it on real data (the DCC launcher)

The split is **off by default** so existing runs reproduce. The dedicated
higher-level launcher turns it on and runs the whole non-circular flow — define
significant electrodes on `P_def`, decode on the disjoint `P_dec` — in one job:

```bash
cd dcc_scripts/decoding
bash submit_decoding_with_electrode_definition_split_dcc.sh
# tune from the environment:
FRAC_DEF=0.6 SEED=1 ALPHA=0.05 STRATA=congruency,switchType,blockType \
    CONDITIONS="stimulus_congruency_by_switch_prop_block_balanced_conditions" \
    bash submit_decoding_with_electrode_definition_split_dcc.sh
```

Output filenames get a `_defsplit` tag so split and non-split runs don't collide,
and the job log prints how many electrodes survived the held-out selector per ROI.
See `docs/analysis_paths.md` §13 and
`docs/decoding_and_electrode_definition_notes.md` §C.

### Acceptance criteria, ticked
- **Disjoint & complete** — `stratified_trial_split` returns index sets that are
  disjoint and together cover every trial once (§1). ✓
- **Balanced** — each stratum is split ~in half on both sides, so neither
  partition is condition- or block-lopsided (§1). ✓
- **Reproducible** — the split is deterministic under `seed` (§1). ✓
- **FDR selection** — responsiveness is judged with Benjamini–Hochberg FDR across
  channels; dead channels are rejected (§3). ✓
- **Removes the bias** — on pure-noise data, selecting on the scored trials
  inflates accuracy above chance while the disjoint split stays at chance (§4). ✓